# Does the rural PHC staffing–IMR association survive socioeconomic controls?

This exploratory state/UT cross-section extends the bivariate analysis by controlling for (1) per-capita NSDP at current prices in 2021–22 and (2) NFHS-5 female literacy among women aged 15–49 in 2019–21. The outcome and staffing indicator are for 2023. All covariates predate the outcome, but their reference years differ. This is an observational, small-sample association—not a causal design.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

DATA = Path('../Dataset')
staff = pd.read_csv(DATA / 'state_rural_phc_doctors_2023.csv')
imr = pd.read_csv(DATA / 'state_infant_mortality_rate_2023.csv')
socio = pd.read_csv(DATA / 'state_socioeconomic_covariates_nfhs5_rbi.csv')
print('Staff rows:', len(staff), '| IMR rows:', len(imr), '| socioeconomic rows:', len(socio))
socio.head()

## 1. Build a transparent complete-case sample

The staffing measure is in-position rural PHC doctors divided by the source-reported required rural PHC doctors. It is not doctors per capita. The outcome is 2023 infant mortality per 1,000 live births. The model controls for log nominal per-capita NSDP (₹, 2021–22) and NFHS-5 female literacy (2019–21). The combined Dadra & Nagar Haveli and Daman & Diu UT and Lakshadweep lack a separate 2021–22 NSDP observation in the RBI table; these two matched observations are excluded from the adjusted complete-case sample rather than imputed. The workforce–IMR match itself does not include Ladakh.

In [ ]:
imr = imr.loc[imr['state_ut'] != 'All India'].copy()
imr['state_key'] = imr['state_ut'].replace({'Dadra & Nagar Haveli (includes Daman & Diu)': 'Dadra & Nagar Haveli & Daman & Diu'})
staff['state_key'] = staff['state_ut']
base = staff.merge(imr[['state_key','infant_mortality_rate_2023_per_1000_live_births']], on='state_key', how='inner', validate='one_to_one')
base['staffing_ratio'] = base['in_position_rural_phc_doctors'] / base['required_rural_phc_doctors'].where(base['required_rural_phc_doctors'] > 0)
base = base.dropna(subset=['staffing_ratio','infant_mortality_rate_2023_per_1000_live_births']).copy()
analysis = base.merge(socio, left_on='state_ut', right_on='state_ut', how='left', validate='one_to_one')
analysis['log_nsdp'] = np.log(pd.to_numeric(analysis['per_capita_nsdp_2021_22_inr'], errors='coerce'))
analysis = analysis.dropna(subset=['female_literacy_nfhs5_pct','log_nsdp']).copy()
print('Bivariate matched sample:', len(base))
print('Adjusted complete-case sample:', len(analysis))
print('Excluded from adjusted model due to missing NSDP:', sorted(set(base.state_ut) - set(analysis.state_ut)))
print('Reference years: IMR/staffing=2023; female literacy=2019–21; NSDP=2021–22')
analysis[['state_ut','staffing_ratio','infant_mortality_rate_2023_per_1000_live_births','per_capita_nsdp_2021_22_inr','female_literacy_nfhs5_pct']].sort_values('state_ut').head(10)

## 2. Compare bivariate and adjusted models on the same sample

The first model is the unadjusted relationship estimated on the exact same complete-case sample as the adjusted model. The second adds both socioeconomic controls. HC3 heteroskedasticity-consistent standard errors are reported for both. The staffing coefficient is the expected difference in IMR (deaths per 1,000 live births) associated with a one-unit increase in the staffing ratio, conditional on the included controls.

In [ ]:
outcome = 'infant_mortality_rate_2023_per_1000_live_births'
model_unadjusted = smf.ols(f'{outcome} ~ staffing_ratio', data=analysis).fit(cov_type='HC3')
model_adjusted = smf.ols(f'{outcome} ~ staffing_ratio + log_nsdp + female_literacy_nfhs5_pct', data=analysis).fit(cov_type='HC3')

def coefficient_table(model, label):
    ci = model.conf_int()
    return pd.DataFrame({
        'model': label,
        'term': model.params.index,
        'coefficient': model.params.values,
        'HC3_std_error': model.bse.values,
        'p_value_HC3': model.pvalues.values,
        'ci95_low': ci.iloc[:,0].values,
        'ci95_high': ci.iloc[:,1].values,
    })
results = pd.concat([coefficient_table(model_unadjusted, 'Unadjusted, common sample'), coefficient_table(model_adjusted, 'Adjusted, HC3')], ignore_index=True)
display(results.round(4))
print('Adjusted model n:', int(model_adjusted.nobs))
print('Adjusted model R-squared:', round(model_adjusted.rsquared, 3))
print('Adjusted model HC3 robust F-test p-value:', model_adjusted.f_pvalue)
print(model_adjusted.summary())

In [ ]:
labels = ['Unadjusted (same 31 states/UTs)', 'Adjusted for NSDP and female literacy']
models = [model_unadjusted, model_adjusted]
estimates = [m.params['staffing_ratio'] for m in models]
lower = [m.conf_int().loc['staffing_ratio', 0] for m in models]
upper = [m.conf_int().loc['staffing_ratio', 1] for m in models]
errors = [[e-l for e,l in zip(estimates, lower)], [u-e for u,e in zip(upper, estimates)]]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.errorbar(estimates, labels, xerr=errors, fmt='o', capsize=4)
ax.axvline(0, linestyle='--', linewidth=1, color='black')
ax.set_xlabel('Staffing-ratio coefficient (IMR deaths per 1,000 live births)')
ax.set_title('Rural PHC staffing coefficient: unadjusted vs adjusted')
fig.tight_layout()
fig.savefig('../Visualisations/11_adjusted_regression_staffing_coefficient.svg', format='svg', bbox_inches='tight')
plt.show()

## 3. Diagnostics and small-denominator sensitivity

With only about 30 observations and three predictors, estimates may be sensitive to individual states. Report variance inflation factors as a basic collinearity diagnostic and re-estimate the adjusted model after excluding states/UTs with fewer than 50 required PHC doctors. The threshold is a sensitivity check, not a formal statistical cutoff. HC3 standard errors do not solve omitted-variable bias, measurement error or influential-observation concerns.

In [ ]:
X = analysis[['staffing_ratio','log_nsdp','female_literacy_nfhs5_pct']].copy()
X = (X - X.mean()) / X.std(ddof=0)
X.insert(0, 'const', 1.0)
vif = pd.DataFrame({'variable': X.columns, 'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
display(vif.round(3))

restricted = analysis.loc[analysis['required_rural_phc_doctors'] >= 50].copy()
print('Restricted complete-case n:', len(restricted))
if len(restricted) > 8:
    model_restricted = smf.ols(f'{outcome} ~ staffing_ratio + log_nsdp + female_literacy_nfhs5_pct', data=restricted).fit(cov_type='HC3')
    print('Restricted staffing coefficient:', round(model_restricted.params['staffing_ratio'], 4))
    print('Restricted HC3 SE:', round(model_restricted.bse['staffing_ratio'], 4))
    print('Restricted HC3 p-value:', round(model_restricted.pvalues['staffing_ratio'], 4))
else:
    print('Restricted sample too small for a stable three-predictor sensitivity model.')

## 4. Interpretation checklist

- Compare the staffing coefficient and HC3 confidence interval before and after controls. A smaller coefficient or wider interval indicates that the bivariate association is partly explained by the included socioeconomic covariates and/or less precisely estimated in the smaller sample.
- The income measure is nominal per-capita NSDP at current prices, not real income or household consumption. State price-level differences are not adjusted.
- NFHS-5 literacy is a survey measure for women aged 15–49; it is not the same construct as the Census literacy rate for all women.
- The model does not control for maternal health, nutrition, sanitation, urbanisation, fertility, health-system quality or other confounders.
- HC3 addresses heteroskedasticity in standard errors; it does not make the model causal or eliminate small-sample sensitivity.
- Treat the estimates as exploratory. Do not claim that staffing changes cause changes in infant mortality.

In [ ]:
# Reproducibility and data-integrity checks
assert len(base) == 34, f'Expected 34 bivariate observations; got {len(base)}'
assert len(analysis) == 32, f'Expected 32 complete-case observations; got {len(analysis)}'
assert analysis[['staffing_ratio','log_nsdp','female_literacy_nfhs5_pct',outcome]].notna().all().all()
assert (analysis['per_capita_nsdp_2021_22_inr'] > 0).all()
assert model_adjusted.nobs == len(analysis)
assert np.isfinite(model_adjusted.params).all()
print('Notebook 07 checks passed.')